# Mistral 7B Fine-tuning on Midjourney Prompt Dataset

This notebook demonstrates fine-tuning Mistral 7B with LoRA on the Midjourney prompt creation dataset.

The fine-tuned model is available at: [PunyaModi/mistral-7b-finetuned-Midjourney-prompt-v2](https://huggingface.co/PunyaModi/mistral-7b-finetuned-Midjourney-prompt-v2)

## What this notebook covers
- Loading and exploring the Midjourney prompt dataset
- Fine-tuning Mistral 7B with QLoRA (4-bit quantization)
- Evaluating the fine-tuned model
- Pushing the adapter to Hugging Face Hub

In [ ]:
!pip install -q torch transformers peft trl bitsandbytes datasets accelerate sentencepiece

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
from lorakit.config import FullConfig, ModelConfig, LoRAConfig, DataConfig, TrainingConfig
from lorakit.data.dataset import DatasetLoader
from lorakit.data.preprocessor import TextPreprocessor, PROMPT_TEMPLATES
from lorakit.training.trainer import LoRATrainer
from lorakit.inference.generator import TextGenerator

print('Ready')

## Step 1: Explore the Dataset

In [ ]:
df = pd.read_csv('../train.csv')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(5)

In [ ]:
print('Sample entry:')
print(df.iloc[0].to_dict())
print(f'\nAvg text length: {df.iloc[:, 0].str.len().mean():.0f} chars')

## Step 2: Configure Fine-tuning

We use QLoRA (4-bit quantization + LoRA) for memory-efficient training.

In [ ]:
config = FullConfig(
    model=ModelConfig(
        model_name='mistralai/Mistral-7B-v0.1',
        use_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype='float16',
        use_nested_quant=False,
    ),
    lora=LoRAConfig(
        r=16,
        lora_alpha=32,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        lora_dropout=0.05,
        bias='none',
        task_type='CAUSAL_LM',
    ),
    data=DataConfig(
        train_file='../train.csv',
        text_column='text',
        max_seq_length=512,
        val_split=0.1,
    ),
    training=TrainingConfig(
        output_dir='../outputs/mistral-midjourney',
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=1,
        learning_rate=2e-4,
        optimizer='paged_adamw_32bit',
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        max_grad_norm=0.3,
        fp16=True,
        bf16=False,
        logging_steps=10,
        save_steps=100,
        eval_steps=100,
        report_to='none',
        push_to_hub=False,
    ),
)

print('Config ready')
print(f'Model: {config.model.model_name}')
print(f'LoRA r={config.lora.r}, alpha={config.lora.lora_alpha}')
print(f'Training for {config.training.num_train_epochs} epochs')

## Step 3: Load Dataset

In [ ]:
loader = DatasetLoader(config.data)
dataset = loader.load()
stats = loader.get_dataset_stats(dataset)
print('Dataset loaded!')
for split, info in stats.items():
    print(f'  {split}: {info["num_examples"]} examples')

## Step 4: Preview Formatted Samples

In [ ]:
preprocessor = TextPreprocessor(config.data)
formatter = preprocessor.get_formatter('raw')
sample = dataset['train'][0]
formatted = formatter(sample)
print('Formatted sample:')
print(formatted['text'][:500])

## Step 5: Fine-tune the Model

Requires a GPU with at least 16GB VRAM for Mistral 7B with 4-bit quantization.

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
trainer = LoRATrainer(config)

logs = []
def on_log(entry):
    logs.append(entry)
    if 'loss' in entry:
        print(f"Step {entry['step']:4d} | Epoch {entry.get('epoch', 0):.2f} | Loss: {entry['loss']:.4f}")

result = trainer.train(dataset, prompt_template='raw', on_log=on_log)
print('\nTraining complete!')
print(f'Final loss: {result["train_loss"]:.4f}')
print(f'Runtime: {result["train_runtime"]:.0f}s')

## Inspect Trainable Parameters

In [ ]:
params = trainer.get_trainable_params()
print(f'Total parameters: {params["total_parameters"]:,}')
print(f'Trainable parameters: {params["trainable_parameters"]:,}')
print(f'Trainable: {params["trainable_percent"]}%')

## Step 6: Generate Midjourney Prompts

In [ ]:
generator = TextGenerator(
    model_path='../outputs/mistral-midjourney',
    base_model='mistralai/Mistral-7B-v0.1',
    load_in_4bit=True,
)
generator.load()

test_inputs = [
    'a futuristic cyberpunk city at night',
    'a serene Japanese garden in autumn',
    'an astronaut exploring an alien planet',
]

for inp in test_inputs:
    outputs = generator.generate(inp, max_new_tokens=150, temperature=0.7)
    print(f'Input: {inp}')
    print(f'Generated: {outputs[0]}')
    print('---')

## Step 7: Push to Hugging Face Hub (Optional)

In [ ]:
from lorakit.models.manager import ModelManager

HF_TOKEN = 'hf_your_token_here'
REPO_ID = 'your-username/mistral-7b-midjourney'

if HF_TOKEN != 'hf_your_token_here':
    manager = ModelManager()
    manager.push_to_hub(
        model_path='../outputs/mistral-midjourney',
        repo_id=REPO_ID,
        token=HF_TOKEN,
        private=False,
    )
    print(f'Model pushed to https://huggingface.co/{REPO_ID}')
else:
    print('Set HF_TOKEN to push to Hub')